In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

print("=== REAL_SOLVE_100_DEPTH300_LOAD_WORLD2_KAGGLE_START ===", flush=True)
print("python", sys.version, flush=True)
subprocess.run(["nvidia-smi", "-L"], check=True)
import torch
print("cuda_available", torch.cuda.is_available(), flush=True)
print("cuda_device_count", torch.cuda.device_count(), flush=True)
assert torch.cuda.device_count() == 2
print(f"cuda_device_0={torch.cuda.get_device_name(0)}", flush=True)
print(f"cuda_device_1={torch.cuda.get_device_name(1)}", flush=True)
assert "T4" in torch.cuda.get_device_name(0)
assert "T4" in torch.cuda.get_device_name(1)

repo_url = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
repo_branch = "codex-architecture-v6-real-data-100-d300-b65536"
PROJECT_DIR = Path("/kaggle/working/CayleyBeam100H100_real_solve_100_d300_w2")
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
print("GIT_CLONE_SOURCE", repo_url, repo_branch, flush=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", repo_branch, repo_url, str(PROJECT_DIR)], check=True)
print("PROJECT_DIR", PROJECT_DIR, flush=True)
required = ["setup.py", "production_v6_dispatcher.py", "data_loader.py", "beam_engine.py", "tests/real_solve_100_depth300_load_world2.py", "data/test.csv", "data/puzzle_info.json", "FullBeamNice/weights/p900-t000-q-sym_1777988767_best.pth", "third_party/cutlass/include/cutlass/gemm/device/gemm.h"]
for rel in required:
    print("required_file", rel, (PROJECT_DIR / rel).exists(), flush=True)
    assert (PROJECT_DIR / rel).exists()

env = os.environ.copy()
env.update({
    "PYTHONUNBUFFERED": "1",
    "CUDA_VISIBLE_DEVICES": "0,1",
    "USE_CUDA_GRAPHS": "1",
    "B_MICRO": "8192",
    "K_EXPAND_TILE": "196608",
    "BUCKET_CAP_PER_PEER": "262144",
    "MAX_DEPTH": "300",
    "TASK_COUNT": "100",
    "GLOBAL_BEAM_WIDTH": "65536",
    "LOG_EACH_DEPTH": "0",
    "LOG_ON_SOLVED": "1",
    "COLLECTIVE_SEQ_DEBUG": "0",
    "TORCH_CUDA_ARCH_LIST": "7.5",
    "MAX_JOBS": "2",
    "INFERENCE_BACKEND": "fullbeamnice_static",
    "FULLBEAMNICE_DIR": str(PROJECT_DIR / "FullBeamNice"),
    "NCCL_IB_DISABLE": "1",
    "NCCL_P2P_DISABLE": "1",
    "NCCL_SOCKET_IFNAME": "lo",
    "GLOO_SOCKET_IFNAME": "lo",
})
b_micro = int(env["B_MICRO"])
k_expand_tile = int(env["K_EXPAND_TILE"])
bucket_cap = int(env["BUCKET_CAP_PER_PEER"])
print("REAL_SOLVE_PRE_TORCHRUN_CONFIG", f"B_MICRO={b_micro}", f"K_EXPAND_TILE={k_expand_tile}", f"BUCKET_CAP_PER_PEER={bucket_cap}", f"USE_CUDA_GRAPHS={env['USE_CUDA_GRAPHS']}", flush=True)
assert b_micro == 8192
assert k_expand_tile == 196608
assert bucket_cap == 262144
assert env["USE_CUDA_GRAPHS"] == "1"

os.chdir(PROJECT_DIR)
print("Step 1: build extension", flush=True)
subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], check=True, env=env)
cmd = [sys.executable, "-u", "-m", "torch.distributed.run", "--standalone", "--nnodes=1", "--nproc_per_node=2", "tests/real_solve_100_depth300_load_world2.py"]
print("Step 2: real solve 100 depth300 WORLD2", flush=True)
print("cmd =", cmd, flush=True)
t0 = time.perf_counter()
p = subprocess.Popen(cmd, cwd=str(PROJECT_DIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert p.stdout is not None
for line in p.stdout:
    print(line, end="", flush=True)
returncode = p.wait()
elapsed_sec = time.perf_counter() - t0
print(f"KAGGLE_NOTEBOOK_WALL_TIME elapsed_sec={elapsed_sec:.3f}", flush=True)
print(f"returncode {returncode}", flush=True)
assert returncode == 0
print("=== REAL_SOLVE_100_DEPTH300_LOAD_WORLD2_TEST_COMPLETE ===", flush=True)
